In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [7]:
df = pd.read_csv('../feed/TrainTestSplit/Risk_Score.csv')
df.head()

,monthly_txn_count,avg_txn_amount,cross_border_ratio,cash_deposit_ratio,account_age_months,risk_score
0,122,940.242412,0.688500,0.411255,74,54.378744
1,199,475.958853,0.058194,0.777602,67,38.469092
2,112,862.194398,0.915214,0.480370,83,58.220420
3,34,1772.396505,0.442352,0.985286,122,63.852578
4,126,682.472791,0.239787,0.376739,125,30.603238


In [8]:
df2 = df.copy()
df2.shape,df.shape

((200, 6), (200, 6))

In [9]:
df2.isna().sum()

monthly_txn_count     0
avg_txn_amount        0
cross_border_ratio    0
cash_deposit_ratio    0
account_age_months    0
risk_score            0
dtype: int64

##### Seperating Feature and Target,    Target = risk_score


In [16]:
features = df2.drop('risk_score',axis=1)
target = df2[['risk_score']]

In [17]:
X = features.copy()
y = target.copy()

In [18]:
X.head()

,monthly_txn_count,avg_txn_amount,cross_border_ratio,cash_deposit_ratio,account_age_months
0,122,940.242412,0.688500,0.411255,74
1,199,475.958853,0.058194,0.777602,67
2,112,862.194398,0.915214,0.480370,83
3,34,1772.396505,0.442352,0.985286,122
4,126,682.472791,0.239787,0.376739,125


In [19]:
y.head()

,risk_score
0,54.378744
1,38.469092
2,58.220420
3,63.852578
4,30.603238


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [22]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
# random_state=42 ensures the same trin and test split every time we run the code. This is important for reproducibility and consistency in results.

In [23]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape,X.shape,y.shape

((160, 5), (40, 5), (160, 1), (40, 1), (200, 5), (200, 1))

In [25]:
X_train.shape[0]/X.shape[0],y_train.shape[0]/y.shape[0]

(0.8, 0.8)

In [26]:
X_test.shape[0]/X.shape[0],y_test.shape[0]/y.shape[0]

(0.2, 0.2)

In [27]:
model = LinearRegression()
model.fit(X_train,y_train)

LinearRegression()

##### Now we use the model in predicting the x_test 20% of the values (features)
##### and we see how close we can predict and get the y_test 20% of the target => model has not seen this data

In [30]:
y_pred = model.predict(X_test)
y_pred[:10]

array([[63.75563937],
       [39.6452719 ],
       [55.26509094],
       [ 9.85596619],
       [67.37644785],
       [58.61525691],
       [43.43490116],
       [47.22400467],
       [74.14495156],
       [67.06067508]])

In [29]:
y_test[:10]

,risk_score
95,66.343679
15,48.583911
30,50.634941
158,8.725277
128,69.174853
115,57.002388
69,38.512584
170,41.572205
174,81.297452
45,75.594228


In [31]:
comparison_df = y_test.copy().rename(columns={"risk_score": "actual"})
comparison_df["predicted"] = y_pred.flatten()
comparison_df["difference"] = comparison_df["actual"] - comparison_df["predicted"]

comparison_df.head(10)

,actual,predicted,difference
95,66.343679,63.755639,2.588039
15,48.583911,39.645272,8.938640
30,50.634941,55.265091,-4.630150
158,8.725277,9.855966,-1.130689
128,69.174853,67.376448,1.798405
115,57.002388,58.615257,-1.612869
69,38.512584,43.434901,-4.922317
170,41.572205,47.224005,-5.651800
174,81.297452,74.144952,7.152501
45,75.594228,67.060675,8.533553


##### In order to calculate the prediction we use Mean Squared Error so the negetives like -4.6 will be (-4.6*-4.6)  and will be posotive 
##### this way we can note the error

In [34]:
from sklearn.metrics import mean_squared_error, r2_score

In [35]:
mse = mean_squared_error(y_test, y_pred)
mse

28.773951754309604

In [37]:
# MSE using the mathematical formula:
# MSE = (1/n) * Σ (y_i - ŷ_i)^2

n = len(y_test)
errors_squared = (y_test["risk_score"].values - y_pred.flatten()) ** 2
mse_manual = (1 / n) * np.sum(errors_squared)

mse_manual

np.float64(28.773951754309607)

In [38]:
# Mean Squared Error (MSE) in simple terms:
# MSE = (1/n) * Σ(actual - predicted)^2
#
# - "actual - predicted" = error for one row
# - square (^2) makes all errors positive and gives bigger mistakes more weight
# - Σ means add all squared errors
# - divide by n (number of rows) to get average error size

print(f"Number of test rows (n): {n}")
print(f"First 5 squared errors: {errors_squared[:5]}")
print(f"MSE (average squared error): {mse:.4f}")

Number of test rows (n): 40
First 5 squared errors: [ 6.69794677 79.89927653 21.43829134  1.27845809  3.23426189]
MSE (average squared error): 28.7740


In [40]:
from IPython.display import display, Markdown

display(Markdown(r"""
### Mean Squared Error (MSE)

The formula for **Mean Squared Error (MSE)** is:

\[
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\]

**In simple terms:**
- \(y_i\): the actual (true) value  
- \(\hat{y}_i\): the predicted value  
- \(n\): the number of data points  
- \((y_i - \hat{y}_i)^2\): the squared error for one data point  

**What it means:**
1. Take the difference between actual and predicted values  
2. Square each difference (so negatives do not cancel positives, and larger errors are penalized more)  
3. Sum all squared errors  
4. Divide by the total number of data points (average error)
"""))


### Mean Squared Error (MSE)

The formula for **Mean Squared Error (MSE)** is:

\[
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\]

**In simple terms:**
- \(y_i\): the actual (true) value  
- \(\hat{y}_i\): the predicted value  
- \(n\): the number of data points  
- \((y_i - \hat{y}_i)^2\): the squared error for one data point  

**What it means:**
1. Take the difference between actual and predicted values  
2. Square each difference (so negatives do not cancel positives, and larger errors are penalized more)  
3. Sum all squared errors  
4. Divide by the total number of data points (average error)


#### Measuring with R2

In [36]:
r2_score(y_test, y_pred)

0.9005856482737534

In [41]:
# R² manual calculation using SST and SSR

y_true = y_test["risk_score"].values
y_hat = y_pred.flatten()

y_mean = y_true.mean()

# SST: Total Sum of Squares -> total variability in actual values around their mean
SST = np.sum((y_true - y_mean) ** 2)

# SSR: Regression Sum of Squares -> variability explained by predictions
SSR = np.sum((y_hat - y_mean) ** 2)

# SSE: Error (Residual) Sum of Squares -> unexplained variability
SSE = np.sum((y_true - y_hat) ** 2)

# Two equivalent forms of R²
r2_from_ssr_sst = SSR / SST
r2_from_sse_sst = 1 - (SSE / SST)

print(f"SST (total variation): {SST:.6f}")
print(f"SSR (explained variation): {SSR:.6f}")
print(f"SSE (unexplained variation): {SSE:.6f}")
print(f"Check: SST ≈ SSR + SSE -> {SST:.6f} ≈ {(SSR + SSE):.6f}\n")

print(f"R² = SSR / SST        = {r2_from_ssr_sst:.6f}")
print(f"R² = 1 - SSE / SST    = {r2_from_sse_sst:.6f}")
print(f"R² (sklearn r2_score) = {r2_score(y_test, y_pred):.6f}")

SST (total variation): 11577.383448
SSR (explained variation): 10436.466298
SSE (unexplained variation): 1150.958070
Check: SST ≈ SSR + SSE -> 11577.383448 ≈ 11587.424368

R² = SSR / SST        = 0.901453
R² = 1 - SSE / SST    = 0.900586
R² (sklearn r2_score) = 0.900586


In [42]:
display(Markdown(r"""
### Simple Definition of R²

**R² tells you how good your model is at explaining the data.**

**Formula (simple idea):**

\[
R^2 = 1 - \frac{\text{errors from your model}}{\text{total variation in data}}
\]

**More formal formula:**

\[
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
\]

### Super simple explanation
- First, imagine just guessing the average every time → that’s your baseline  
- Then compare it to your model’s predictions  

👉 **R² measures how much better your model is than just guessing the average.**

### Easy way to think about it
- **R² = 1** → perfect predictions  
- **R² = 0** → same as guessing the average  
- **R² = 0.7** → your model explains 70% of what’s happening  

### One-line summary
**R² = how much of the data your model successfully explains (in percentage form).**
"""))


### Simple Definition of R²

**R² tells you how good your model is at explaining the data.**

**Formula (simple idea):**

\[
R^2 = 1 - \frac{\text{errors from your model}}{\text{total variation in data}}
\]

**More formal formula:**

\[
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
\]

### Super simple explanation
- First, imagine just guessing the average every time → that’s your baseline  
- Then compare it to your model’s predictions  

👉 **R² measures how much better your model is than just guessing the average.**

### Easy way to think about it
- **R² = 1** → perfect predictions  
- **R² = 0** → same as guessing the average  
- **R² = 0.7** → your model explains 70% of what’s happening  

### One-line summary
**R² = how much of the data your model successfully explains (in percentage form).**


In [43]:
# # Simple synthetic banking/AML dataset for Multiple Linear Regression
# np.random.seed(42)

# n_samples = 200

# monthly_txn_count = np.random.randint(20, 250, size=n_samples)
# avg_txn_amount = np.random.uniform(50, 2000, size=n_samples)
# cross_border_ratio = np.random.uniform(0.0, 1.0, size=n_samples)   # 0 to 1
# cash_deposit_ratio = np.random.uniform(0.0, 1.0, size=n_samples)   # 0 to 1
# account_age_months = np.random.randint(6, 180, size=n_samples)

# # Linear target (AML risk score) with small noise
# noise = np.random.normal(0, 5, size=n_samples)
# aml_risk_score = (
#     0.08 * monthly_txn_count
#     + 0.015 * avg_txn_amount
#     + 35 * cross_border_ratio
#     + 25 * cash_deposit_ratio
#     - 0.05 * account_age_months
#     + noise
# )

# banking_aml_df = pd.DataFrame({
#     "monthly_txn_count": monthly_txn_count,
#     "avg_txn_amount": avg_txn_amount,
#     "cross_border_ratio": cross_border_ratio,
#     "cash_deposit_ratio": cash_deposit_ratio,
#     "account_age_months": account_age_months,
#     "aml_risk_score": aml_risk_score
# })

# # Features/target for multiple linear regression
# X = banking_aml_df.drop(columns=["aml_risk_score"])
# y = banking_aml_df["aml_risk_score"]

# # Optional quick fit to verify linear learnability
# model = LinearRegression()
# model.fit(X, y)

# print(banking_aml_df.head())
# print("\nR^2 on training data:", round(model.score(X, y), 4))

In [4]:
from pathlib import Path

feed_dir = Path("feed")
feed_dir.mkdir(parents=True, exist_ok=True)

csv_path = feed_dir / "banking_aml_dataset.csv"
excel_path = feed_dir / "banking_aml_dataset.xlsx"

banking_aml_df.to_csv(csv_path, index=False)
banking_aml_df.to_excel(excel_path, index=False)

print(f"Saved CSV to: {csv_path.resolve()}")
print(f"Saved Excel to: {excel_path.resolve()}")

Saved CSV to: C:\Users\Lenovo\OneDrive\Desktop\Projects\Data Science_2\Linear models\Supervised Learning\feed\banking_aml_dataset.csv
Saved Excel to: C:\Users\Lenovo\OneDrive\Desktop\Projects\Data Science_2\Linear models\Supervised Learning\feed\banking_aml_dataset.xlsx


In [39]:
from IPython.display import display, Markdown

display(Markdown(r"""
### Mean Squared Error (MSE)

The formula for **Mean Squared Error (MSE)** is:

\[
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\]

**In simple terms:**
- \(y_i\): the actual (true) value  
- \(\hat{y}_i\): the predicted value  
- \(n\): the number of data points  
- \((y_i - \hat{y}_i)^2\): the squared error for one data point  

**What it means:**
1. Take the difference between actual and predicted values  
2. Square each difference (so negatives do not cancel positives, and larger errors are penalized more)  
3. Sum all squared errors  
4. Divide by the total number of data points (average error)
"""))


### Mean Squared Error (MSE)

The formula for **Mean Squared Error (MSE)** is:

\[
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\]

**In simple terms:**
- \(y_i\): the actual (true) value  
- \(\hat{y}_i\): the predicted value  
- \(n\): the number of data points  
- \((y_i - \hat{y}_i)^2\): the squared error for one data point  

**What it means:**
1. Take the difference between actual and predicted values  
2. Square each difference (so negatives do not cancel positives, and larger errors are penalized more)  
3. Sum all squared errors  
4. Divide by the total number of data points (average error)
